In [1]:
import torch

text = "hello world"
# Get all unique characters to form our vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create mapping dictionaries: char -> int and int -> char
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encode function: string to list of integers
def encode(s):
    return [stoi[c] for c in s]

# Decode function: list of integers to string
def decode(l):
    return ''.join([itos[i] for i in l])

token_ids = encode(text)
print(f"Vocabulary: {chars}")
print(f"Vocab size: {vocab_size}")
print(f"Encoded '{text}': {token_ids}")
print(f"Decoded: {decode(token_ids)}")

Vocabulary: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
Vocab size: 8
Encoded 'hello world': [3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1]
Decoded: hello world


In [2]:
import torch.nn as nn
import math

# We will represent each character with a 16-dimensional vector (d_model)
d_model = 16 

# The embedding layer maps our vocabulary size to the chosen dimension
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)

# Convert our list of token IDs into a PyTorch tensor with a batch dimension
# Shape: (Batch Size=1, Sequence Length=11)
input_tensor = torch.tensor([token_ids])

# Pass tokens through the embedding layer
# Scaling the embeddings by the square root of the dimensionality is a standard practice.
token_embeddings = embedding_layer(input_tensor) * math.sqrt(d_model)

print(f"Input tensor shape: {input_tensor.shape}")
print(f"Embedding tensor shape: {token_embeddings.shape}") 
# Output shape will be: (1, 11, 16) -> (batch_size, sequence_length, d_model)

Input tensor shape: torch.Size([1, 11])
Embedding tensor shape: torch.Size([1, 11, 16])


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        # Initialize a tensor to hold the positional encodings
        pe = torch.zeros(max_len, d_model)
        
        # Calculate sine and cosine frequencies
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sin to even dimensions and cos to odd dimensions
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add a batch dimension and register as a non-trainable buffer
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add the positional encoding vector to the token embedding vector
        # We slice the buffer up to the sequence length of the input 'x'
        x = x + self.pe[:, :x.size(1), :]
        return x

# Initialize positional encoding
pos_encoder = PositionalEncoding(d_model=d_model, max_len=50)

# Add positional data to our word embeddings
final_input_representation = pos_encoder(token_embeddings)

print(f"Final input shape ready for Transformer: {final_input_representation.shape}")

Final input shape ready for Transformer: torch.Size([1, 11, 16])
